# Retail Orders — Executable Data Profile & Quality Contract

This notebook profiles completeness, uniqueness, validity, consistency, and freshness against the agreed data-quality contract. It intentionally fails the current sample because the source contains quality issues.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

ORDERS_FILE = Path('retail-orders-raw (1).csv')
orders = pd.read_csv(ORDERS_FILE)
orders.head()


In [ ]:
# Normalize fields used by the contract.
orders['order_id'] = orders['order_id'].astype('string').str.strip()
orders['customer_segment_norm'] = orders['customer_segment'].astype('string').str.strip().str.title()
orders['category_norm'] = orders['category'].astype('string').str.strip()
orders['payment_status_norm'] = orders['payment_status'].astype('string').str.strip().str.title()
orders['quantity_num'] = pd.to_numeric(orders['quantity'], errors='coerce')
orders['discount_num'] = pd.to_numeric(orders['discount_pct'], errors='coerce')
orders['order_date_parsed'] = pd.to_datetime(orders['order_date'], errors='coerce')
orders


In [ ]:
# Define the approved domains and reusable checks.
allowed_segments = {'Student','Fresher','Professional'}
allowed_categories = {'Learning Kit','Course Access','Mentor Session'}
allowed_statuses = {'Paid','Pending','Failed','Refunded'}
today = pd.Timestamp.today().normalize()
   
checks = []
def add_check(check_id, dimension, rule, passed, total, threshold, action):
    rate = (passed / total * 100) if total else 100.0
    status = 'PASS' if rate >= threshold else 'FAIL'
    checks.append([check_id, dimension, rule, total, passed, round(rate,2), threshold, status, action])

n = len(orders)
add_check('Q-01','Completeness','Required fields populated',
          orders[['order_id','order_date','customer_segment','city','category','quantity','unit_price','payment_status']].notna().all(axis=1).sum(), n, 100, 'Block KPI publication; fix source data.')
add_check('Q-02','Discount completeness','Missing discount <= 5%',
          orders['discount_num'].notna().sum(), n, 95, 'Hold pricing KPIs; obtain business approval before treating missing as zero.')
add_check('Q-03','Uniqueness','order_id has no duplicates',
          orders['order_id'].nunique(), n, 100, 'Block order-level KPIs; investigate duplicate records.')
add_check('Q-04','Validity','order_date parses and is within 2025-01-01 to today',
          orders['order_date_parsed'].between(pd.Timestamp('2025-01-01'), today, inclusive='both').fillna(False).sum(), n, 100, 'Block time-series KPIs; repair invalid dates.')
add_check('Q-05','Validity','quantity is a positive integer',
          ((orders['quantity_num'].notna()) & (orders['quantity_num'] > 0) & (orders['quantity_num'] % 1 == 0)).sum(), n, 100, 'Block sales and volume KPIs.')
add_check('Q-06','Validity','unit_price is non-negative',
          (orders['unit_price'] >= 0).sum(), n, 100, 'Block sales KPIs.')
add_check('Q-07','Validity','discount is between 0 and 100',
          orders['discount_num'].between(0,100, inclusive='both').fillna(False).sum(), n, 100, 'Block net-sales and discount KPIs.')
add_check('Q-08','Domain validity','approved categorical domains',
          (orders['customer_segment_norm'].isin(allowed_segments) & orders['category_norm'].isin(allowed_categories) & orders['payment_status_norm'].isin(allowed_statuses)).sum(), n, 100, 'Quarantine invalid categories/statuses and notify data owner.')
quality = pd.DataFrame(checks, columns=['Rule ID','Dimension','Rule','Rows','Passing','Pass Rate %','Threshold %','Status','Action'])
quality


In [ ]:
# Consistency checks: derived monetary fields must be reproducible.
valid_numeric = orders['quantity_num'].gt(0) & orders['unit_price'].ge(0) & orders['discount_num'].between(0,100)
orders['gross_value'] = orders['quantity_num'] * orders['unit_price']
orders['net_value'] = orders['gross_value'] * (1 - orders['discount_num']/100)
consistency = pd.DataFrame({
    'gross_recalculable': orders.loc[valid_numeric, 'gross_value'].notna(),
    'net_recalculable': orders.loc[valid_numeric, 'net_value'].notna()
})
print('Consistency pass rate:', round(consistency.all(axis=1).mean()*100,2), '%')


In [ ]:
# Freshness check. In production, compare max source date with the expected refresh date.
max_source_date = orders['order_date_parsed'].max()
print('Latest valid order date:', max_source_date)
print('Freshness rule: data should be no more than 1 business day behind the expected refresh date.')
print('This sample has no extract timestamp, so freshness is documented but cannot be fully automated from the source alone.')


In [ ]:
# Final contract gate.
blocking_rules = {'Q-01','Q-03','Q-04','Q-05','Q-06','Q-07','Q-08'}
failed_blockers = quality.loc[quality['Rule ID'].isin(blocking_rules) & (quality['Status']=='FAIL'), 'Rule ID'].tolist()
if failed_blockers:
    raise AssertionError(f'DATA QUALITY CONTRACT FAILED: {failed_blockers}')
print('DATA QUALITY CONTRACT PASSED')
